## AMCL Performance Analysis

In [86]:
import math
import numpy as np
import pandas as pd
from rosbags.rosbag2 import Reader

"""
Analysis Outline
Collected Topics
- /amcl_pose, /particle_cloud, /odometry/filtered
Scenerio
- took 2 turns around 3 waypoints, stopping for 5 seconds at each waypoint after the waypoint is reached

Waypoint coordinates relative to the map frame in detail

w1->(1.18, 2.52, pi/2)
w2->(-0.75, 2.57, pi/2)
w3->(0,0,pi/2)

Tasks
->Capture the arrival times for all the 6 points
->store /amcl_pose + /odometry/filtered estimates for all the points / (x,y,theta).
->calculate statistics: mean absolute error, mean_squared_error, estimates
"""

GROUND_TRUTHS = {
    "w1": (1.18, 2.52, math.pi/2),
    "w2": (-0.75, 2.57, math.pi/2),
    "w3": ((0, 0, math.pi/2))
}

# --- CONFIGURATION ---
BAG_PATH = "/home/rosgeek/Documents/lab_parkur_ws/src/lab_mapping/rosbags/rosbag2_2026_04_11-13_33_05_0"  # Folder containing the .db3
TOPIC = "/amcl_pose"

In [87]:
from pathlib import Path
from rosbags.highlevel import AnyReader
from rosbags.typesys import Stores, get_typestore
import datetime

class Pose:
     @classmethod
     def get_average(self, poses):
          n = len(poses)
          sum_pose = Pose()
          for pose in poses:
               sum_pose.x+=pose.x
               sum_pose.y+=pose.y
               sum_pose.theta+=pose.theta
          sum_pose.x /= n
          sum_pose.y /= n
          sum_pose.theta /= n

          return sum_pose

     def __init__(self, x=0.0,y=0.0,theta=0.0,time=0.0):
          self.x = x
          self.y = y
          self.theta = theta
          self.time = time
          
     def __repr__(self):
          return f"x: {self.x}, y: {self.y}, theta: {self.theta}, time: {self.time}"
     
def quaternion_to_yaw(qx, qy, qz, qw):
    """Converts 2D planar quaternion to Euler Yaw (theta) in radians."""
    siny_cosp = 2 * (qw * qz + qx * qy)
    cosy_cosp = 1 - 2 * (qy * qy + qz * qz)
    return math.atan2(siny_cosp, cosy_cosp)

def timestamp_to_hour_minute(timestamp):
     return datetime.datetime.fromtimestamp(timestamp).strftime('%H:%M:%S')

In [88]:
bagpath = Path(BAG_PATH)
typestore = get_typestore(Stores.ROS2_FOXY)

amcl_poses = [] #Pose array
odom_filtered_speeds = []


# Create reader instance and open for reading.
with AnyReader([bagpath], default_typestore=typestore) as reader:
    connections = [x for x in reader.connections if x.topic == '/amcl_pose']
    for connection, timestamp, rawdata in reader.messages(connections=connections):
         msg = reader.deserialize(rawdata, connection.msgtype)
         yaw = quaternion_to_yaw(msg.pose.pose.orientation.x, 
                                 msg.pose.pose.orientation.y, 
                                 msg.pose.pose.orientation.z, 
                                 msg.pose.pose.orientation.w)
         x,y = msg.pose.pose.position.x, msg.pose.pose.position.y
         time = msg.header.stamp.sec + msg.header.stamp.nanosec / 1e9
         pose = Pose(x,y,yaw,time)
         amcl_poses.append(pose)

    connections = [x for x in reader.connections if x.topic == '/odometry/filtered']
    for connection, timestamp, rawdata in reader.messages(connections=connections):
         msg = reader.deserialize(rawdata, connection.msgtype)
         vx, vy = msg.twist.twist.linear.x, msg.twist.twist.linear.y
         v = np.sqrt(vx**2 + vy**2)

         time = msg.header.stamp.sec + msg.header.stamp.nanosec / 1e9
         odom_filtered_speeds.append((v, time))

for speed, time in odom_filtered_speeds:
     print(f"speed: {speed}, time: {time}")

speed: 0.0, time: 1775914386.3935506
speed: 0.0, time: 1775914386.4239473
speed: 0.0, time: 1775914386.4534552
speed: 0.0, time: 1775914386.4930768
speed: 0.0, time: 1775914386.5134878
speed: 0.0, time: 1775914386.5533645
speed: 0.0, time: 1775914386.5931695
speed: 0.0, time: 1775914386.613467
speed: 0.0, time: 1775914386.6532786
speed: 0.0, time: 1775914386.693116
speed: 0.0, time: 1775914386.7237923
speed: 0.0, time: 1775914386.7532976
speed: 0.0, time: 1775914386.7931223
speed: 0.0, time: 1775914386.8135622
speed: 0.0, time: 1775914386.8532684
speed: 0.0, time: 1775914386.894146
speed: 0.0, time: 1775914386.913445
speed: 0.0, time: 1775914386.9532971
speed: 0.0, time: 1775914386.9941158
speed: 0.0, time: 1775914387.0134468
speed: 0.0, time: 1775914387.0532825
speed: 0.0, time: 1775914387.0941815
speed: 0.0, time: 1775914387.1238112
speed: 0.0, time: 1775914387.1533473
speed: 0.0, time: 1775914387.2658968
speed: 0.0, time: 1775914387.3668764
speed: 0.0, time: 1775914387.3957052
speed

In [89]:
# Take moving average of the odom_filtered measurements
moving_sum = 0
window = round(6 / 0.04) # Moving average over 6 seconds
speed_averages = []

for i in range(len(odom_filtered_speeds)):
    if i % window != 0:
        moving_sum += odom_filtered_speeds[i][0] 
    else:
        avg_speed = moving_sum / window
        time = timestamp_to_hour_minute(round(odom_filtered_speeds[i][1]))
        speed_averages.append((avg_speed, time))
        moving_sum = 0


In [90]:
for avg, time in speed_averages:
    print(f"avg: {avg}, time: {time}")

avg: 0.0, time: 16:33:06
avg: 0.0, time: 16:33:12
avg: 0.0, time: 16:33:18
avg: 0.0, time: 16:33:24
avg: 0.15927711950533494, time: 16:33:30
avg: 0.26451556107289553, time: 16:33:35
avg: 0.18181516416843643, time: 16:33:41
avg: 0.09606584520060922, time: 16:33:47
avg: 0.0800143922445779, time: 16:33:53
avg: 0.014649039404128556, time: 16:33:59
avg: 0.10538850617724768, time: 16:34:05
avg: 0.09948702395779539, time: 16:34:11
avg: 0.12354584142795193, time: 16:34:17
avg: 0.09666428004338452, time: 16:34:22
avg: 0.14067322095472543, time: 16:34:28
avg: 0.31678311541239046, time: 16:34:34
avg: 0.23895848492765454, time: 16:34:40
avg: 0.170896949105394, time: 16:34:46
avg: 0.004778452345919961, time: 16:34:52
avg: 0.1461202778567155, time: 16:34:58
avg: 0.1976466435074662, time: 16:35:03
avg: 0.17139988183316934, time: 16:35:09
avg: 0.0003353027003673613, time: 16:35:15
avg: 0.00927379128813729, time: 16:35:21
avg: 0.34061646927275896, time: 16:35:27
avg: 0.10237659967984085, time: 16:35:33

### Stop Times Using the Simulation Data
- w1.1->avg: 0.014649039404128556, time: 16:33:59, 1775914439
- w2.1->avg: 0.004778452345919961, time: 16:34:52, 1775914492
- w3.1->avg: 0.00040344668136232297, time: 16:36:02, 1775914562
- w1.2->avg: 0.0003458376561566371, time: 16:36:31, 1775914591
- w2.2->avg: 0.035179545875399766, time: 16:37:13, 1775914633
- w3.2->avg: 0.00023344786333710625, time: 16:38:41, 1775914721

In [91]:
for pose in amcl_poses:
    print(pose.__repr__())

x: 0.14480015116930764, y: 1.5246251717303783, theta: 2.2794571362571103, time: 1775914374.9197757
x: 0.14966186145420526, y: 1.501689804997896, theta: 1.7822741162821656, time: 1775914408.1369565
x: 0.17020885919567041, y: 1.4951148557592637, theta: 1.4894630143169305, time: 1775914408.6412094
x: 0.15323996522917033, y: 1.4917225823420552, theta: 1.2261426385933087, time: 1775914409.0569904
x: 0.1568080612643929, y: 1.482349281710053, theta: 1.0672110999957696, time: 1775914409.4807749
x: 0.1344166866515926, y: 1.4688531095568058, theta: 1.1263260618978668, time: 1775914409.9042256
x: 0.1183826190707513, y: 1.4795463917120995, theta: 1.043815036211663, time: 1775914410.7449515
x: 0.13335148005554096, y: 1.500879622655452, theta: 1.289038888832908, time: 1775914411.2487054
x: 0.13438591485568496, y: 1.5412261647902206, theta: 1.2562248727893865, time: 1775914412.5934932
x: 0.1854847522878792, y: 1.6090586736695118, theta: 1.2710879533176322, time: 1775914414.02406
x: 0.3061004651866499

## AMCL Poses + Differences


In [105]:
waypoint_times = {
    "w1_1": 1775914439,
    "w1_2": 1775914591,
    "w2_1": 1775914492,
    "w2_2": 1775914633,
    "w3_1": 1775914562,
    "w3_2": 1775914721
}

waypoint_poses = {
    wp: [] for wp in waypoint_times.keys()
}

#Filter amcl_pose estimations at waypoint timestamps, take the averages at each point
for wp, time in waypoint_times.items():
    if wp == "w3_2":
        waypoint_poses[wp] = Pose.get_average(amcl_poses[-3:])
    else:
        wp_estimates = [pose for pose in amcl_poses if abs(pose.time - time) < 7]
        waypoint_poses[wp] = Pose.get_average(wp_estimates)


for wp, estimate in waypoint_poses.items():
    print(f"{wp}: {estimate.__repr__()}")

w1_1: x: 1.2423772495395509, y: 1.99602361397758, theta: 2.156674626743815, time: 0.0
w1_2: x: 1.334406073220486, y: 1.71118467168039, theta: 2.313141074244669, time: 0.0
w2_1: x: -0.7740814168162646, y: 2.531839601201153, theta: 0.7918185778173373, time: 0.0
w2_2: x: -0.7765898050802513, y: 2.2579026014654118, theta: 1.4806281983616432, time: 0.0
w3_1: x: -0.01351125246836524, y: 0.28308392629167767, theta: 1.1944621025276025, time: 0.0
w3_2: x: -0.11539721358062677, y: 0.08255904437247144, theta: 1.5801215224142322, time: 0.0


In [ ]:
GROUND_TRUTHS = {
    "w1": (1.18, 2.52, math.pi/2),
    "w2": (-0.75, 2.57, math.pi/2),
    "w3": ((0, 0, math.pi/2))
}

0.8972036348145294

## Numpy Manipulation of the Metrics

In [108]:
import numpy as np

# 1. Initialize the measurements and ground truths as numpy arrays
# Structure: [x, y, theta]
estimates = np.array([
    [1.2423772495395509, 1.99602361397758, 2.156674626743815],   # w1_1
    [1.334406073220486, 1.71118467168039, 2.313141074244669],    # w1_2
    [-0.7740814168162646, 2.531839601201153, 0.7918185778173373],# w2_1
    [-0.7765898050802513, 2.2579026014654118, 1.4806281983616432],# w2_2
    [-0.01351125246836524, 0.28308392629167767, 1.1944621025276025],# w3_1
    [-0.11539721358062677, 0.08255904437247144, 1.5801215224142322] # w3_2
])

ground_truths = np.array([
    [1.18, 2.52, 1.5707963],  # w1_1
    [1.18, 2.52, 1.5707963],  # w1_2
    [-0.75, 2.57, 1.5707963], # w2_1
    [-0.75, 2.57, 1.5707963], # w2_2
    [0.0, 0.0, 1.5707963],    # w3_1
    [0.0, 0.0, 1.5707963]     # w3_2
])

labels = ["w1_1", "w1_2", "w2_1", "w2_2", "w3_1", "w3_2"]

# 2. Calculate Individual Absolute Errors
# Separate x, y from theta for proper angular wrapping
diff_xy = estimates[:, :2] - ground_truths[:, :2]
abs_err_xy = np.abs(diff_xy)

raw_diff_theta = estimates[:, 2] - ground_truths[:, 2]
abs_err_theta = np.abs(np.arctan2(np.sin(raw_diff_theta), np.cos(raw_diff_theta)))

# Recombine into a single array of shape (6, 3)
individual_abs_errors = np.column_stack((abs_err_xy, abs_err_theta))

# 3. Calculate Means across all waypoints (axis=0 calculates the mean of each column)
mean_amcl_estimate = np.mean(estimates, axis=0)
mean_ground_truth = np.mean(ground_truths, axis=0)

# 4. Calculate Mean Absolute Error (MAE)
mean_absolute_error = np.mean(individual_abs_errors, axis=0)

# 5. Calculate Mean Squared Error (MSE)
# Square the individual absolute errors, then find the mean
mean_squared_error = np.mean(individual_abs_errors**2, axis=0)

# --- Format and Print the Results ---
print("--- INDIVIDUAL ABSOLUTE ERRORS ---")
print(f"{'Point':<6} | {'X Error (m)':<12} | {'Y Error (m)':<12} | {'Theta Error (rad)':<15}")
print("-" * 55)
for i, label in enumerate(labels):
    print(f"{label:<6} | {individual_abs_errors[i, 0]:<12.6f} | {individual_abs_errors[i, 1]:<12.6f} | {individual_abs_errors[i, 2]:<15.6f}")

print("\n--- AGGREGATE METRICS ---")
print(f"{'Metric':<25} | {'X':<12} | {'Y':<12} | {'Theta':<12}")
print("-" * 60)
print(f"{'Mean AMCL Estimate':<25} | {mean_amcl_estimate[0]:<12.6f} | {mean_amcl_estimate[1]:<12.6f} | {mean_amcl_estimate[2]:<12.6f}")
print(f"{'Mean Ground Truth':<25} | {mean_ground_truth[0]:<12.6f} | {mean_ground_truth[1]:<12.6f} | {mean_ground_truth[2]:<12.6f}")
print(f"{'Mean Absolute Error (MAE)':<25} | {mean_absolute_error[0]:<12.6f} | {mean_absolute_error[1]:<12.6f} | {mean_absolute_error[2]:<12.6f}")
print(f"{'Mean Squared Error (MSE)':<25} | {mean_squared_error[0]:<12.6f} | {mean_squared_error[1]:<12.6f} | {mean_squared_error[2]:<12.6f}")

--- INDIVIDUAL ABSOLUTE ERRORS ---
Point  | X Error (m)  | Y Error (m)  | Theta Error (rad)
-------------------------------------------------------
w1_1   | 0.062377     | 0.523976     | 0.585878       
w1_2   | 0.154406     | 0.808815     | 0.742345       
w2_1   | 0.024081     | 0.038160     | 0.778978       
w2_2   | 0.026590     | 0.312097     | 0.090168       
w3_1   | 0.013511     | 0.283084     | 0.376334       
w3_2   | 0.115397     | 0.082559     | 0.009325       

--- AGGREGATE METRICS ---
Metric                    | X            | Y            | Theta       
------------------------------------------------------------
Mean AMCL Estimate        | 0.149534     | 1.477099     | 1.586141    
Mean Ground Truth         | 0.143333     | 1.696667     | 1.570796    
Mean Absolute Error (MAE) | 0.066061     | 0.341449     | 0.430505    
Mean Squared Error (MSE)  | 0.007086     | 0.185758     | 0.275163    
